# Notebook 13: Scale Comparison and the ID→OOD Gap

**Purpose**: Add the missing ID query split and a 70B arm. Settles Notebook 11's
saturation confound, tests Gate S1(a), and overturns the premise of RQ1.

## What this run adds

`random x balanced` at corruption 0/50/100% plus zero-shot, **both ID and OOD
query splits**, k=8, 5 seeds x 250 queries, at `Llama-3.1-8B-Instruct` and
`Llama-3.1-70B-Instruct` (fp8). 128 units / 32,000 rows per scale.

```bash
cd sata-project
# 8B on GPU0, both splits, calibration on
PYTHONPATH=. python scripts/run_real_arm_grid.py --mode corruption-gate \
    --cache-dir _screen_cache --model Llama-3.1-8B-Instruct \
    --out results/v2/gate_s0c_full --query-split both \
    --tensor-parallel 1 --max-model-len 4096
# 70B fp8 on GPU1 -- fits one H200 at fp8, so TP=1 and the two run concurrently
PYTHONPATH=. python scripts/run_real_arm_grid.py --mode corruption-gate \
    --cache-dir _screen_cache --model Llama-3.1-70B-Instruct \
    --out results/v2/gate_s0c_70b --query-split both \
    --tensor-parallel 1 --quantization fp8 --max-model-len 4096
```

Two questions this answers that no previous notebook could:

1. **Is there actually an ID→OOD gap?** Every prior run used OOD queries only, so
   the project's central quantity had never been measured.
2. **Is the label channel inert, or just saturated?** A 70B model with a much
   wider output dynamic range discriminates the two hypotheses Notebook 11 could
   not separate and Notebook 12 could only partially address.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

pd.set_option("display.width", 220)
RESULTS = PROJECT_ROOT / "results" / "v2"


def margin(df):
    """Label-logprob margin: the model's decision variable before thresholding."""
    return df.logprob_1 - df.logprob_0


def p_positive(df):
    """Implied P(positive) = sigmoid(margin)."""
    return 1.0 / (1.0 + np.exp(df.logprob_0 - df.logprob_1))


def integrity(df, name=""):
    """A failed label-token lookup defaults to -100 and a failed decode to -1.
    Both would masquerade as findings, so check before interpreting anything."""
    bad_lp = ((df.logprob_0 == -100) | (df.logprob_1 == -100)).mean()
    bad_pred = (df.prediction_raw == -1).mean()
    print(f"{name}invalid predictions: {bad_pred:.5f} | logprob sentinels: {bad_lp:.5f}")
    return bad_lp == 0 and bad_pred == 0

In [3]:
# Schema note: the GPU runner records the query split in `environment` ("id"/"ood"),
# not in a `query_split` column -- that name belongs to the CPU screen's schema
# (Notebook 10). Easy to trip over when moving code between the two.
R8 = pd.read_parquet(RESULTS / "gate_s0c_8b_merged.parquet")
R70 = pd.read_parquet(RESULTS / "gate_s0c_70b_merged.parquet")
R = pd.concat([R8.assign(scale="8B"), R70.assign(scale="70B")], ignore_index=True)
assert integrity(R8, "8B: ") and integrity(R70, "70B: ")
print(f"{len(R)} rows | splits={sorted(R.environment.unique())} | scales={R.scale.unique().tolist()}")

8B: invalid predictions: 0.00000 | logprob sentinels: 0.00000
70B: invalid predictions: 0.00000 | logprob sentinels: 0.00000
64000 rows | splits=['id', 'ood'] | scales=['8B', '70B']


## Step 1: The headline — there is no ID→OOD gap

Paired over seeds, within each (scale, dataset), at 0% corruption.

In [4]:
rows = []
for (sc, ds), g in R[(R.mechanism == "random") & (R.corruption == 0.0)].groupby(["scale", "dataset"]):
    per = {sp: gs.groupby("seed").apply(lambda x: roc_auc_score(x.label, margin(x)))
           for sp, gs in g.groupby("environment")}
    aid, aood = per["id"].to_numpy(), per["ood"].to_numpy()
    rows.append(dict(scale=sc, dataset=ds, auroc_id=aid.mean(), auroc_ood=aood.mean(),
                     gap=aid.mean() - aood.mean(), seeds_id_higher=int((aid > aood).sum()),
                     wilcoxon_p=stats.wilcoxon(aid, aood).pvalue))
GP = pd.DataFrame(rows).sort_values(["scale", "dataset"])
print(GP.round(3).to_string(index=False))
print(f"\nmean |gap| = {GP.gap.abs().mean():.3f} | max |gap| = {GP.gap.abs().max():.3f}")
print(f"cells where ID > OOD: {(GP.gap > 0).sum()} of {len(GP)} | "
      f"any cell significant at 0.05: {(GP.wilcoxon_p < 0.05).any()}")

scale        dataset  auroc_id  auroc_ood    gap  seeds_id_higher  wilcoxon_p
  70B      acsincome     0.823      0.841 -0.018                1       0.188
  70B      acspubcov     0.582      0.527  0.055                5       0.062
  70B           anes     0.731      0.730  0.001                2       1.000
  70B brfss_diabetes     0.758      0.780 -0.023                2       0.812
   8B      acsincome     0.676      0.682 -0.006                3       1.000
   8B      acspubcov     0.476      0.517 -0.042                2       0.312
   8B           anes     0.710      0.682  0.028                3       0.438
   8B brfss_diabetes     0.596      0.592  0.004                2       0.812

mean |gap| = 0.022 | max |gap| = 0.055
cells where ID > OOD: 4 of 8 | any cell significant at 0.05: False


**There is no ID→OOD generalisation gap for LLM ICL on these datasets, at
either scale.** The sign splits evenly, no cell is significant, and the effect is
an order of magnitude smaller than the shifts are real.

The mechanism is clean in hindsight: **an ICL model is never *fitted* to the
source distribution, so it has no source-specific decision boundary to be wrong
about in the target domain.** The shifts genuinely exist — Notebook 09 measured
domain-discriminator AUCs up to 0.92 — and Notebook 09's supervised
HistGradientBoosting baseline *does* degrade on exactly these splits. The LLM
does not, because there is nothing in it that was tuned to the source.

**Consequence for RQ1.** "How much does ICL degrade under distribution shift, and
can demonstration design reduce that degradation?" presupposes degradation that
does not occur here. The honest reframing is comparative:

> *ICL trades absolute in-domain accuracy for shift-invariance. A supervised
> model beats it in-domain and loses that advantage under shift; the ICL model is
> flat across the shift. Where is the crossover, and can demonstration design
> raise the ICL level without reintroducing shift-sensitivity?*

That is answerable with data already collected, and it is a more interesting
claim than the original.

**Power caveat, cutting the other way.** With 5 seeds this null is *itself*
underpowered — the honest form is "any gap is smaller than this design can
resolve", not "the gap is zero". The contrast against the supervised baseline is
what carries the argument, and that contrast does not depend on a significance
test. See Notebook 14 §4.

## Step 2: Scale kills the saturation confound

In [5]:
sat = R[(R.mechanism == "random") & (R.corruption == 0.0) & (R.environment == "ood")]
S = sat.assign(pyes=p_positive).groupby(["scale", "dataset"]).pyes.agg(
    median="median", iqr=lambda v: np.diff(np.percentile(v, [25, 75]))[0]).round(3)
print(S.to_string())
print("\nmean IQR by scale:")
print(S.groupby(level=0).iqr.mean().round(3).to_string())

                      median    iqr
scale dataset                      
70B   acsincome        0.905  0.673
      acspubcov        0.867  0.247
      anes             0.893  0.583
      brfss_diabetes   0.706  0.799
8B    acsincome        0.867  0.087
      acspubcov        0.867  0.058
      anes             0.835  0.174
      brfss_diabetes   0.924  0.035

mean IQR by scale:
scale
70B    0.576
8B     0.088


The 70B model's decision variable has a **far wider dynamic range** — mean IQR
roughly 6.5x the 8B model's. Notebook 11's "no room left to register an effect"
hypothesis predicts the label channel should become visible here.

## Step 3: It does not. The label channel is inert at 70B too.

In [6]:
rows = []
for (sc, ds), g in R[(R.mechanism == "random") & (R.environment == "ood")].groupby(["scale", "dataset"]):
    per = {c: gc.groupby("seed").apply(lambda x: roc_auc_score(x.label, margin(x)))
           for c, gc in g.groupby("corruption")}
    a0, a1 = per[0.0].to_numpy(), per[1.0].to_numpy()
    rows.append(dict(scale=sc, dataset=ds, auroc_c0=a0.mean(), auroc_c100=a1.mean(),
                     delta=a0.mean() - a1.mean(), seeds_same_dir=int((a0 > a1).sum()),
                     wilcoxon_p=stats.wilcoxon(a0, a1).pvalue))
pd.DataFrame(rows).sort_values(["scale", "delta"], ascending=[True, False]).round(3)

,scale,dataset,auroc_c0,auroc_c100,delta,seeds_same_dir,wilcoxon_p
2,70B,anes,0.730,0.602,0.128,5,0.062
3,70B,brfss_diabetes,0.780,0.731,0.049,3,0.312
0,70B,acsincome,0.841,0.827,0.014,4,0.125
1,70B,acspubcov,0.527,0.547,-0.020,2,0.438
6,8B,anes,0.682,0.507,0.175,5,0.062
4,8B,acsincome,0.682,0.670,0.012,3,0.625
5,8B,acspubcov,0.517,0.518,-0.001,3,1.000
7,8B,brfss_diabetes,0.592,0.617,-0.025,2,0.625


ANES remains the only dataset that responds, at both scales. Everything else is
flat, now with a decision variable that has ample room to move.

**This definitively settles the confound.** Combined with Notebook 12 §3 (three
metrics agreeing) and the de-saturation attempt failing, the conclusion is that
these models are not learning the input→label mapping from tabular
demonstrations — not that the measurement was blind to it.

**Consequence for RQ2.** Any mechanism that works by choosing *which labels* the
model sees has a near-zero ceiling on 3 of 4 datasets, at both scales. That does
**not** rule out the *feature* channel — see Notebook 14.

## Step 4: Calibration is sign-inverted at 70B

In [7]:
cal = R[(R.mechanism == "random") & (R.corruption == 0.0) &
        (R.environment == "ood")].groupby(["scale", "dataset"]).apply(lambda s: pd.Series({
    "observed_margin": float(margin(s).mean()),
    "content_free_margin": float((s.logprob_1_cf - s.logprob_0_cf).mean()),
    "posrate_raw": (s.prediction_raw == 1).mean(),
    "posrate_calibrated": (s.prediction == 1).mean(),
    "bacc_raw": balanced_accuracy_score(s.label, s.prediction_raw),
    "bacc_calibrated": balanced_accuracy_score(s.label, s.prediction),
    "auroc": roc_auc_score(s.label, margin(s)),
})).round(3)
cal

observed_margin  content_free_margin  posrate_raw  posrate_calibrated  bacc_raw  bacc_calibrated  auroc
scale dataset                                                                                                                
70B   acsincome                 2.075               -5.409        0.690               0.983     0.703            0.514  0.838
      acspubcov                 1.863               -1.382        0.880               0.947     0.516            0.515  0.514
      anes                      1.971               -4.512        0.709               0.956     0.644            0.532  0.725
      brfss_diabetes            0.254               -5.810        0.594               0.971     0.697            0.518  0.801
8B    acsincome                 1.891                1.941        0.995               0.426     0.502            0.664  0.678
      acspubcov                 1.859                1.694        1.000               0.777     0.500            0.503  0.519
      anes                      1.263                1.794        0.927               0.427     0.519            0.637  0.678
      brfss_diabetes            2.479                2.272        1.000               0.794     0.500            0.562  0.548

**A method finding that matters for every future run.** At 8B the content-free
margin is positive and tracks the observed margin, so calibration works. At 70B
it is strongly *negative* — so calibration *adds* up to 5.8 to every margin,
pushes the calibrated positive rate to 0.95-0.98, and **collapses calibrated
balanced accuracy to ~0.52 while raw AUROC is at its best (0.84)**.

**Use calibrated predictions at 8B and raw predictions at 70B.** Applying a
single decision rule across scales would have produced a completely false
"70B is worse" conclusion. Notebook 14 therefore skips calibration for its 70B
arm entirely — which also halves that arm's prompt count.

## Step 5: Demonstrations as an implicit calibrator

In [8]:
# NB: `scale` is a grouping key, so pandas excludes it from the group frame --
# read it from the key, not from `s.scale`, or this raises AttributeError.
rows = []
for (sc, ds, mech), g in R[(R.corruption == 0.0) & (R.environment == "ood")].groupby(
        ["scale", "dataset", "mechanism"]):
    pred = g.prediction if sc == "8B" else g.prediction_raw
    rows.append(dict(scale=sc, dataset=ds, mechanism=mech,
                     auroc=roc_auc_score(g.label, margin(g)),
                     bacc=balanced_accuracy_score(g.label, pred)))
(pd.DataFrame(rows).pivot_table(index=["scale", "dataset"], columns="mechanism",
                                values=["auroc", "bacc"]).round(3))

auroc             bacc          
mechanism            random zero_shot random zero_shot
scale dataset                                         
70B   acsincome       0.838     0.840  0.703     0.731
      acspubcov       0.514     0.587  0.516     0.551
      anes            0.725     0.743  0.644     0.603
      brfss_diabetes  0.801     0.720  0.697     0.661
8B    acsincome       0.678     0.708  0.664     0.448
      acspubcov       0.519     0.489  0.503     0.500
      anes            0.678     0.716  0.637     0.499
      brfss_diabetes  0.548     0.675  0.562     0.517

At 8B, adding demonstrations buys **+0.14 to +0.22 balanced accuracy over
zero-shot with essentially no AUROC gain** — they are not teaching the task, they
are shifting the decision threshold into a usable place. At 70B that service is
redundant: zero-shot is already well-centred, and few-shot adds nothing (or
slightly less).

This reframes what demonstrations are *for* in tabular ICL at small scale: an
implicit calibration device, not a learning signal. It also predicts that
protocols which work *through* the threshold should beat protocols that try to
teach the mapping — which is what Notebook 14 tests.

## Step 6: Gate S1(a), now testable

The gate requires random-8 ID accuracy >= 0.60. The right comparison is not the
fixed 0.60 but the **majority-class rate** — on BRFSS, 88% of rows are negative,
so always guessing "no" scores 0.88 and a 0.60 bar is *below* chance.

In [9]:
rows = []
for (sc, ds), g in R[(R.mechanism == "random") & (R.corruption == 0.0) &
                     (R.environment == "id")].groupby(["scale", "dataset"]):
    pred = g.prediction if sc == "8B" else g.prediction_raw
    maj = max(g.label.mean(), 1 - g.label.mean())
    acc = (pred == g.label).mean()
    rows.append(dict(scale=sc, dataset=ds, acc_id=acc, majority_id=maj,
                     passes_fixed_060=acc >= 0.60, beats_majority=acc > maj))
G = pd.DataFrame(rows).sort_values(["scale", "dataset"])
print(G.round(3).to_string(index=False))
print(f"\ncells beating majority-class: {G.beats_majority.sum()} of {len(G)}")

scale        dataset  acc_id  majority_id  passes_fixed_060  beats_majority
  70B      acsincome   0.598        0.685             False           False
  70B      acspubcov   0.370        0.778             False           False
  70B           anes   0.710        0.718              True           False
  70B brfss_diabetes   0.506        0.877             False           False
   8B      acsincome   0.672        0.685              True           False
   8B      acspubcov   0.401        0.778             False           False
   8B           anes   0.591        0.718             False           False
   8B brfss_diabetes   0.284        0.877             False           False

cells beating majority-class: 0 of 8


**No (dataset x scale) cell beats its majority-class baseline.** Several pass the
literal 0.60 bar, which is exactly why that bar is the wrong instrument.

**Gate S1 must be rewritten** to require beating majority-class prevalence (or to
use balanced accuracy > 0.5 + margin) before it is used as a go/no-go anywhere.
As written it is not a competence floor on imbalanced data.